# Eco-City PPO — Google Colab (GPU)

1. **Runtime → Change runtime type → GPU** (e.g. A100 if available).
2. Run the **clone** cell below once per session. It pulls the repo from GitHub and sets `PROJECT_ROOT`.

Repository: [https://github.com/Shreya-Mendi/Eco-city](https://github.com/Shreya-Mendi/Eco-city)

In [ ]:
# Clone the repo from GitHub (Colab) and move to project root
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Shreya-Mendi/Eco-city.git"
CLONE_PARENT = Path("/content")
PROJECT_ROOT = CLONE_PARENT / "Eco-city"

if not (PROJECT_ROOT / "requirements.txt").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Repo already exists; pulling latest...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("CWD:", os.getcwd())

## Hyperparameters (edit these)

| Setting | Default | Notes |
|---------|---------|--------|
| `TOTAL_TIMESTEPS` | 500_000 | Smoke test: 50_000–100_000 |
| `SAVE_PATH` | `results/ppo_eco_city` | SB3 writes `SAVE_PATH.zip` |
| `DEVICE` | `"auto"` | Colab GPU: usually `"cuda"`; `"cpu"` to force CPU |
| `EXPORT_JSON` | `True` | Writes `city_data.json` for the Three.js viewer |

In [ ]:
# Optional: persist to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/Eco-city")
# os.chdir(PROJECT_ROOT)
# sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from env.city_env import CityEnv
from stable_baselines3.common.env_checker import check_env

_env = CityEnv()
check_env(_env, warn=True)
print("ENV OK")

In [ ]:
from pathlib import Path

from agents.ppo_agent import make_ppo_agent, save, train
from training.callbacks import MetricsCallback
from training.vec_env import make_vec_train_env, save_vecnormalize, unwrap_city_env, vecnorm_save_path
from training.train import _rollout_one_episode_for_export
from visualization.exporter import export

# --- knobs ---
TOTAL_TIMESTEPS = 500_000
LOGDIR = Path("logs/ppo")
SAVE_PATH = "results/ppo_eco_city"
DEVICE = "auto"  # "cuda" | "cpu" | "auto"
EXPORT_JSON = True

LOGDIR.mkdir(parents=True, exist_ok=True)
Path("results").mkdir(parents=True, exist_ok=True)
Path("logs").mkdir(parents=True, exist_ok=True)

vec_env = make_vec_train_env(log_csv=LOGDIR / "monitor.csv")
base = unwrap_city_env(vec_env)

device_kw = {}
if DEVICE != "auto":
    device_kw["device"] = DEVICE

model = make_ppo_agent(vec_env, tensorboard_log="logs", **device_kw)
callback = MetricsCallback()

train(model, total_timesteps=TOTAL_TIMESTEPS, callback=callback)
save(model, SAVE_PATH)
save_vecnormalize(vec_env, vecnorm_save_path(SAVE_PATH))

if EXPORT_JSON:
    base._export_traj = []
    _rollout_one_episode_for_export(model, vec_env, seed=42)
    export(base, "visualization/threejs/city_data.json")
    base._export_traj = None

print("Done. Model:", SAVE_PATH + ".zip")
print("VecNormalize:", vecnorm_save_path(SAVE_PATH))

## Download artifacts (Colab)
Run after training. Download **both** the policy zip and the VecNormalize stats (evaluation needs both).

In [ ]:
try:
    from google.colab import files
    from training.vec_env import vecnorm_save_path
    save = "results/ppo_eco_city"
    files.download(save + ".zip")
    files.download(str(vecnorm_save_path(save)))
    files.download("visualization/threejs/city_data.json")
except ImportError:
    print("Not on Colab — artifacts are under results/ and visualization/threejs/")

## TensorBoard (optional)
In Colab: `%load_ext tensorboard` then `%tensorboard --logdir logs`